In [ ]:
import json
from datetime import datetime

# ===== 1. ระบบจัดการไฟล์ข้อมูล (JSON) =====
def load_json(filename, default_data):
    try:
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
            return {int(k): v for k, v in data.items()} if isinstance(data, dict) else data
    except Exception:
        return default_data

def save_json(filename, data):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# ข้อมูลเริ่มต้นกรณีไม่มีไฟล์
default_menu = {
    1: {"name": "ข้าวผัดกุ้ง", "price": 60, "category": "อาหารคาว"},
    2: {"name": "กะเพราหมูกรอบ", "price": 65, "category": "อาหารคาว"},
    3: {"name": "ต้มยำกุ้ง", "price": 120, "category": "อาหารคาว"},
    4: {"name": "ชาไทยเย็น", "price": 45, "category": "เครื่องดื่ม"},
    5: {"name": "กาแฟอเมริกาโน่", "price": 50, "category": "เครื่องดื่ม"},
}

default_toppings = {
    1: {"name": "🍳 ไข่ดาว", "price": 10},
    2: {"name": "🧀 ชีส", "price": 15},
    3: {"name": "🧋 ไข่มุก", "price": 10},
}

menu = load_json("menu.json", default_menu)
toppings = load_json("toppings.json", default_toppings)

# ===== 2. ฟังก์ชันแสดงผลและคำนวณเงิน =====
def display_menu():
    print("\n=== 🍽️ รายการเมนูทั้งหมด ===")
    for k, v in menu.items():
        print(f" [{k}] {v['name']} ({v['category']}) - {v['price']} บาท")

def display_toppings():
    print("\n--- 🍳 ท็อปปิ้งเสริม ---")
    for k, v in toppings.items():
        print(f"  [{k}] {v['name']} (+{v['price']} บาท)")

def calculate_bill(cart):
    subtotal = sum(item['total_price'] for item in cart)
    discount = subtotal * 0.10 if subtotal >= 300 else 0  # ส่วนลด 10% เมื่อสั่งครบ 300 บาท
    after_discount = subtotal - discount
    vat = after_discount * 0.07
    grand_total = after_discount + vat
    return subtotal, discount, vat, grand_total

# ===== 3. ลูปการทำงานหลัก (Main Workflow Loop) =====
def main():
    cart = []  # List เก็บรายการสินค้าในตะกร้า
    customer_name = input("👤 กรุณากรอกชื่อลูกค้า / เลขโต๊ะ: ").strip() or "ลูกค้าทั่วไป"

    while True:
        print(f"\n==========================================")
        print(f"🛒 ระบบสั่งอาหาร | โต๊ะ: {customer_name}")
        print(f"📦 จำนวนสินค้าในตะกร้า: {len(cart)} รายการ")  # [List Operation: len()]
        print("1. ดูเมนูอาหาร & ค้นหาเมนู")
        print("2. เลือกสั่งสินค้าลงตะกร้า")
        print("3. ดูตะกร้า / ลบรายการ / เรียงลำดับราคา")
        print("4. ชำระเงิน & พิมพ์ใบเสร็จ")
        print("5. ออกจากระบบ / จบการทำงาน")
        print("==========================================")

        choice = input("👉 เลือกทำรายการ (1-5): ").strip()

        # --- 1. ดูเมนู & ค้นหา ---
        if choice == "1":
            display_menu()
            display_toppings()

            # [List/String Operation: in (ค้นหา)]
            search = input("\n🔍 พิมพ์ชื่อเมนูที่ต้องการค้นหา (เว้นว่างเพื่อข้าม): ").strip()
            if search:
                found = False
                print(f"\n🔎 ผลการค้นหาสำหรับ '{search}':")
                for k, v in menu.items():
                    if search.lower() in v['name'].lower():  # เช็คคำค้นหาในชื่อเมนู
                        print(f"  [{k}] {v['name']} ({v['category']}) - {v['price']} บาท")
                        found = True
                if not found:
                    print("❌ ไม่พบเมนูที่ตรงกับคำค้นหา")

        # --- 2. สั่งสินค้า ---
        elif choice == "2":
            display_menu()
            try:
                item_id = int(input("\nกรอกรหัสเมนูที่ต้องการสั่ง: "))
                if item_id not in menu:
                    print("❌ ไม่พบรหัสเมนูนี้")
                    continue

                qty = int(input("จำนวนที่ต้องการสั่ง: "))
                if qty <= 0:
                    print("❌ จำนวนต้องมากกว่า 0")
                    continue

                level = input("ระดับความหวาน/ความเผ็ด (เช่น ปกติ, หวานน้อย, เผ็ดมาก): ").strip() or "ปกติ"

                display_toppings()
                top_choice = input("เลือกรหัสท็อปปิ้ง (เว้นว่างหากไม่ต้องการ): ").strip()

                selected_topping = None
                topping_price = 0
                if top_choice.isdigit() and int(top_choice) in toppings:
                    top_info = toppings[int(top_choice)]
                    selected_topping = top_info['name']
                    topping_price = top_info['price']

                unit_price = menu[item_id]['price'] + topping_price
                total_price = unit_price * qty

                # [List Operation: append()]
                cart.append({
                    "name": menu[item_id]['name'],
                    "qty": qty,
                    "level": level,
                    "topping": selected_topping,
                    "unit_price": unit_price,
                    "total_price": total_price
                })
                print(f"✅ เพิ่ม '{menu[item_id]['name']}' ลงตะกร้าเรียบร้อย!")

            except ValueError:
                print("❌ กรุณากรอกข้อมูลเป็นตัวเลขให้ถูกต้อง")

        # --- 3. ดูตะกร้า / ลบรายการ / เรียงลำดับ ---
        elif choice == "3":
            if not cart:
                print("\n🛒 ตะกร้าสินค้ายังว่างอยู่")
            else:
                print(f"\n🛒 --- รายการในตะกร้าของ {customer_name} (ทั้งหมด {len(cart)} รายการ) ---")
                for i, item in enumerate(cart, 1):
                    top_txt = f" + {item['topping']}" if item['topping'] else ""
                    print(f" {i}. {item['name']}{top_txt} [{item['level']}] x{item['qty']} = {item['total_price']} บาท")

                subtotal, discount, vat, grand_total = calculate_bill(cart)
                print(f"\nรวม: {subtotal:.2f} บ. | ส่วนลด: -{discount:.2f} บ. | VAT 7%: {vat:.2f} บ.")
                print(f"💰 ยอดรวมสุทธิ: {grand_total:.2f} บาท")

                # ตัวเลือกเพิ่มเติมสำหรับจัดการ List ในตะกร้า
                print("\n--- 🛠️ ตัวเลือกจัดการตะกร้า ---")
                print(" [A] ลบรายการสินค้าออกจากตะกร้า")
                print(" [B] เรียงลำดับสินค้าในตะกร้าตามราคาสูง-ต่ำ")
                print(" [Enter] ย้อนกลับเมนูหลัก")

                sub_choice = input("👉 เลือกทำรายการ: ").strip().upper()

                # [List Operation: pop() / ลบรายการตามตำแหน่ง]
                if sub_choice == "A":
                    try:
                        remove_idx = int(input(f"กรอกลำดับรายการที่ต้องการลบ (1-{len(cart)}): "))
                        if 1 <= remove_idx <= len(cart):
                            removed_item = cart.pop(remove_idx - 1)  # ลบข้อมูลออกจาก List
                            print(f"🗑️ ลบ '{removed_item['name']}' ออกจากตะกร้าเรียบร้อย!")
                        else:
                            print("❌ ลำดับรายการไม่ถูกต้อง")
                    except ValueError:
                        print("❌ กรุณากรอกตัวเลขลำดับให้ถูกต้อง")

                # [List Operation: sort() / เรียงลำดับ]
                elif sub_choice == "B":
                    cart.sort(key=lambda x: x['total_price'], reverse=True)
                    print("✨ เรียงลำดับรายการในตะกร้าตามราคาสูง-ต่ำ เรียบร้อย!")

        # --- 4. พิมพ์ใบเสร็จ & จบออเดอร์ ---
        elif choice == "4":
            if not cart:
                print("\n⚠️ ตะกร้ายังว่างอยู่ ไม่สามารถชำระเงินได้")
                continue

            subtotal, discount, vat, grand_total = calculate_bill(cart)
            now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            print("\n==========================================")
            print("         🧾 ใบเสร็จรับเงิน (Receipt)")
            print("==========================================")
            print(f"👤 ลูกค้า: {customer_name}")
            print(f"🕒 เวลา: {now_str}")
            print(f"📦 จำนวนรายการ: {len(cart)} รายการ")
            print("------------------------------------------")
            for item in cart:
                top_txt = f" + {item['topping']}" if item['topping'] else ""
                print(f"- {item['name']}{top_txt} ({item['level']}) x{item['qty']} = {item['total_price']:.2f} บาท")
            print("------------------------------------------")
            print(f"ยอดรวมสินค้า:         {subtotal:.2f} บาท")
            print(f"ส่วนลด (10% ครบ 300บ.): -{discount:.2f} บาท")
            print(f"ภาษี VAT (7%):        {vat:.2f} บาท")
            print(f"💰 ยอดชำระสุทธิ:       {grand_total:.2f} บาท")
            print("==========================================")

            # บันทึกลงไฟล์ orders.json
            orders_history = load_json("orders.json", [])
            orders_history.append({
                "timestamp": now_str,
                "customer": customer_name,
                "items": cart,
                "grand_total": grand_total
            })
            save_json("orders.json", orders_history)

            print("🎉 บันทึกออเดอร์และเช็คบิลเรียบร้อย ขอบคุณครับ!")
            cart.clear()  # ล้างตะกร้าเตรียมรับออเดอร์ถัดไป

            next_customer = input("\n👤 กรอกชื่อลูกค้าคนถัดไป/เลขโต๊ะใหม่ (กด Enter เพื่อใช้ชื่อเดิม): ").strip()
            if next_customer:
                customer_name = next_customer

        # --- 5. ออกจากลูป ---
        elif choice == "5":
            print("\n👋 ขอบคุณที่ใช้บริการ ปิดระบบเรียบร้อย")
            break

        else:
            print("❌ เลือกรายการไม่ถูกต้อง กรุณาใส่ตัวเลข 1-5")

if __name__ == "__main__":
    main()

👤 กรุณากรอกชื่อลูกค้า / เลขโต๊ะ: 4

🛒 ระบบสั่งอาหาร | โต๊ะ: 4
📦 จำนวนสินค้าในตะกร้า: 0 รายการ
1. ดูเมนูอาหาร & ค้นหาเมนู
2. เลือกสั่งสินค้าลงตะกร้า
3. ดูตะกร้า / ลบรายการ / เรียงลำดับราคา
4. ชำระเงิน & พิมพ์ใบเสร็จ
5. ออกจากระบบ / จบการทำงาน
👉 เลือกทำรายการ (1-5): 2

=== 🍽️ รายการเมนูทั้งหมด ===
 [1] ข้าวผัดกุ้ง (อาหารคาว) - 60 บาท
 [2] กะเพราหมูกรอบ (อาหารคาว) - 65 บาท
 [3] ต้มยำกุ้ง (อาหารคาว) - 120 บาท
 [4] ชาไทยเย็น (เครื่องดื่ม) - 45 บาท
 [5] กาแฟอเมริกาโน่ (เครื่องดื่ม) - 50 บาท

กรอกรหัสเมนูที่ต้องการสั่ง: 2
จำนวนที่ต้องการสั่ง: 3
ระดับความหวาน/ความเผ็ด (เช่น ปกติ, หวานน้อย, เผ็ดมาก): ปกติ

--- 🍳 ท็อปปิ้งเสริม ---
  [1] 🍳 ไข่ดาว (+10 บาท)
  [2] 🧀 ชีส (+15 บาท)
  [3] 🧋 ไข่มุก (+10 บาท)
เลือกรหัสท็อปปิ้ง (เว้นว่างหากไม่ต้องการ): 1
✅ เพิ่ม 'กะเพราหมูกรอบ' ลงตะกร้าเรียบร้อย!

🛒 ระบบสั่งอาหาร | โต๊ะ: 4
📦 จำนวนสินค้าในตะกร้า: 1 รายการ
1. ดูเมนูอาหาร & ค้นหาเมนู
2. เลือกสั่งสินค้าลงตะกร้า
3. ดูตะกร้า / ลบรายการ / เรียงลำดับราคา
4. ชำระเงิน & พิมพ์ใบเสร็จ
5. ออกจากระบบ / จบการทำงาน
👉 เล